# ecDNA prevalence by tumor type

ecDNA prevalence: fraction of tumors with ecDNA, by tumor type, drawn as rings.

- **Unit:** one tumor per patient and tumor type (`in_unique_tumor_set`). A tumor is ecDNA+ if any of its biosamples is (`amplicon_class == 'ecDNA'`; the dedup puts ecDNA+ biosamples first). Validation cohorts (S.T. 7) are deduplicated the same way, by their `unique_tumor_set` flag, and count toward the types in `VALIDATION_TYPES`.
- **Confidence intervals:** Wilson score intervals (Brown, Cai & DasGupta 2001). Tumors within a type are one per patient, so they are independent binomial trials.
- **Rings:** the ecDNA+ arc runs clockwise from 12 o'clock in the dark shade of the tumor type's color, and the remainder is in the light shade. The CI is an arc with end bars through the middle of the ring. Ring area is proportional to ln(n), at the same scale as the hand-drawn pies in the previous Fig. 3.
- **Molecular subtypes:** the same rings per subclass (`cancer_subclass`) within each tumor type, in the type's colors, with undrawn subclasses pooled into one "Other" ring per type.
- **Panels:** the brain and child panels are composed from `config/prevalence_panel_layout.json`, which sets the rings drawn, their positions, label sides and leader lines, and the silhouettes from `assets/`.
- **Output:** one SVG and PNG per tumor type in `out/prevalence/` and per subtype in `out/prevalence/subtypes/`, the composed panels `panel_brain` and `panel_child`, all with editable Arial text, plus tables of all tumor types and subtypes with CIs.

Replaces the per-type counts in `summary_statistics.ipynb` ("ecDNA+ tumors by type and subtype") and the pie sizing in `sandbox.ipynb` ("Pie chart circle size").

## TODO
- molecular subtypes
- Map TUMOR_TYPE_NAMES from S.T. 12 Abbreviations
- Automated image composition onto a Fig.a,b

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Arc
from matplotlib.transforms import Bbox
from statsmodels.stats.proportion import proportion_confint
sys.path.append('../../src')
from data_imports import *
from seaborn_helper_functions import set_plot_defaults, get_palette
set_plot_defaults()

## Parameters
Tumor type and subtype colors are read from `config/colors.yml`. `TUMOR_TYPE_NAMES` is a plain dict of strings so it can move to a shared config file later.

In [ ]:
# Per tumor type: dark shade (ecDNA+) and light shade (remainder), from config/colors.yml
TUMOR_TYPE_COLORS = get_palette('tumor_type')
DEFAULT_COLORS = TUMOR_TYPE_COLORS.pop('default') # tumor types without an entry

# Display names, with the line breaks used in the panels
TUMOR_TYPE_NAMES = {
    'HGG':  'High-grade glioma',
    'LGG':  'Low-grade glioma',
    'GNT':  'Glioneuronal\ntumors',
    'CPG':  'Craniopharyn-\ngioma',
    'ATRT': 'Atypical teratoid\nrhabdoid tumors',
    'ETMR': 'Embryonal tumor\nwith multilayered\nrosettes*',
    'EMBT': 'Miscellaneous embryonal\nbrain tumors',
    'CPT':  'Choroid plexus\ntumors',
    'PINT': 'Pineal tumors',
    'MBL':  'Medulloblastoma',
    'EPN':  'Ependymoma',
    'MEL':  'Melanoma',
    'PNST': 'Peripheral nerve\nsheath tumors',
    'LIV':  'Tumors of the liver',
    'NBL':  'Neuroblastoma',
    'EWS':  'Ewing sarcoma',
    'RBL':  'Retinoblastoma',
    'RMS':  'Rhabdomyo-\nsarcoma',
    'ACC':  'Adrenocortical\ncarcinoma',
    'WT':   'Wilms tumor',
    'GCT':  'Germ cell tumors',
    'OS':   'Osteosarcoma',
    'BENG': 'Miscellaneous rare\nbenign tumors',
    'SARC': 'Miscellaneous\nrare sarcomas',
    'MST':  'Metastatic\nsecondary tumors',
}

# Panel layouts: ring positions, label sides, leader lines and silhouettes, in mm from each panel's top left
PANEL_LAYOUT = '../../config/prevalence_panel_layout.json'
with open(PANEL_LAYOUT) as f:
    panel_layout = json.load(f)
# Tumor types drawn in the panels (layout order) and the side of the ring their label goes on, read from the layout
LABEL_SIDE = {k: r['label'] for p in panel_layout['panels'].values() for k, r in p['rings'].items() if '.' not in k}

# Tumor types whose ring includes the validation cohorts (S.T. 7; marked * in the figure). The MYCN-amplified EPN
# cohort was selected for MYCN amplification, so it counts toward its own subtype ring but not toward EPN as a whole.
VALIDATION_TYPES = ['ETMR']

# Molecular subtypes. Subtype keys are 'cancer_type.cancer_subclass', eg. 'EPN.MYCN'.
# Subclasses drawn as their own rings, per tumor type. Types not listed draw every subclass with at least MIN_SUBTYPE_N
# tumors. A type's remaining subclasses, and its tumors without a subclass, are pooled into one OTHER_SUBTYPE ring.
DRAWN_SUBTYPES = {}      # eg. {'MBL': ['WNT', 'SHH', 'G3', 'G4']}
MIN_SUBTYPE_N = 10
OTHER_SUBTYPE = 'Other'
# Subtypes whose ring includes the validation cohorts (in addition to VALIDATION_TYPES). Always drawn as their own ring,
# so that selected validation tumors never enter OTHER_SUBTYPE.
VALIDATION_SUBTYPES = ['EPN.MYCN']
# Subtype rings use their tumor type's colors unless overridden in config/colors.yml (tumor_subtype)
SUBTYPE_COLORS = get_palette('tumor_subtype')
# Display names; default is the subclass code
SUBTYPE_NAMES = {'EPN.MYCN': 'MYCN-amplified\nspinal EPN*'}

# Rings (lengths in mm, line widths in points)
DIAMETER_SF = 2.876531550448852 # diameter = DIAMETER_SF * sqrt(ln n); same scale as the previous Fig. 3 pies
RING_WIDTH = 1.0                # ring thickness, constant across sizes
MIN_RADIUS = 1.5                # floor on the outer radius (about n = 3); below that the hole closes, and n = 1 has radius 0
CI_COLOR = 'black'              # 'white' reads better on dark rings
CI_LINEWIDTH = 0.5
CI_TICK = 0.6                   # end bar length, as a fraction of RING_WIDTH
LABEL_GAP = 1.5                 # between ring and label; the layout's label_gap overrides it in composed panels

# Panel composition
LEADER_LINEWIDTH = 0.75         # pt
LETTER_FONTSIZE = 8             # pt, panel letters
INKSCAPE = '/Applications/Inkscape.app/Contents/MacOS/inkscape' # renders composed SVGs to PNG; None to skip

# Confidence intervals
CI_METHOD = 'wilson'            # any statsmodels proportion_confint method, eg. 'beta' (Clopper-Pearson), 'jeffreys'
ALPHA = 0.05

OUTDIR = 'out/prevalence'
os.makedirs(OUTDIR, exist_ok=True)

## Data

In [ ]:
biosamples = import_biosamples()
validation_biosamples = import_validation_biosamples()

In [ ]:
def get_tumors(biosamples, validation_biosamples=None):
    '''
    One row per tumor, with ecDNA status, from each cohort's unique tumor set: the main cohort (in_unique_tumor_set;
    one per patient x tumor type) and, optionally, the validation cohorts (S.T. 7, unique_tumor_set; one per patient).
    In both, the representative biosample is ecDNA+ if any of the tumor's biosamples is.
    Returns: pd.DataFrame with patient_id, cancer_type, cancer_subclass, amplicon_class, cohort ('main' or
    'validation') and ecDNA (bool).
    '''
    cols = ['patient_id', 'cancer_type', 'cancer_subclass', 'amplicon_class']
    tumors = biosamples.loc[biosamples.in_unique_tumor_set, cols].assign(cohort='main')
    if validation_biosamples is not None:
        validation = validation_biosamples.loc[validation_biosamples.unique_tumor_set, cols].assign(cohort='validation')
        tumors = pd.concat([tumors, validation])
    tumors['ecDNA'] = tumors.amplicon_class == 'ecDNA'
    return tumors.reset_index(drop=True)

def prevalence(tumors, by='cancer_type', method=CI_METHOD, alpha=ALPHA):
    '''
    Fraction of tumors that are ecDNA+, with binomial confidence intervals.
    Inputs:
        tumors: pd.DataFrame from get_tumors().
        by: column (or list of columns) to group tumors by, eg. 'cancer_type' or ['cancer_type','cancer_subclass'].
        method, alpha: passed to statsmodels proportion_confint.
    Returns: pd.DataFrame indexed by group, with k (ecDNA+), n, fraction, ci_low, ci_high.
    '''
    df = tumors.groupby(by).ecDNA.agg(k='sum', n='size')
    df['fraction'] = df.k / df.n
    ci = proportion_confint(df.k, df.n, alpha=alpha, method=method)
    df['ci_low'], df['ci_high'] = [np.clip(x, 0, 1) for x in ci] # Wilson can return eg. -1e-17 at k = 0
    return df.sort_values('n', ascending=False)

In [ ]:
tumors = get_tumors(biosamples, validation_biosamples)
# validation tumors count only toward the types in VALIDATION_TYPES
panel_tumors = tumors[(tumors.cohort == 'main') | tumors.cancer_type.isin(VALIDATION_TYPES)]
by_type = prevalence(panel_tumors)
by_type.to_csv(f'{OUTDIR}/prevalence_by_cancer_type.tsv', sep='\t')
by_type

In [ ]:
# contribution of each cohort to the types with validation tumors
prevalence(tumors[tumors.cancer_type.isin(validation_biosamples.cancer_type)], by=['cancer_type', 'cohort'])

In [ ]:
# MYCN-amplified EPN, main and validation cohorts together (subtype ring; colors to be decided with the other subtypes)
mycn_epn = prevalence(tumors[(tumors.cancer_type == 'EPN') & (tumors.cancer_subclass == 'MYCN')],
                      by=['cancer_type', 'cancer_subclass'])
mycn_epn

## Rings

In [ ]:
def get_colors(cancer_type, colors=TUMOR_TYPE_COLORS, default=DEFAULT_COLORS):
    '''Dark and light shades for a tumor type.'''
    return dict(colors.get(cancer_type, default))

def ring_radius(n, sf=DIAMETER_SF, min_radius=MIN_RADIUS):
    '''Outer radius (mm) of a ring whose area is proportional to ln(n), with a floor of min_radius.'''
    return max(sf * np.sqrt(np.log(n)) / 2, min_radius)

def clock_angle(p):
    '''Fraction p of a circle, clockwise from 12 o'clock, as a matplotlib angle (degrees counterclockwise from 3 o'clock).'''
    return 90 - 360 * p

def draw_ring(ax, xy, k, n, ci, dark, light, width=RING_WIDTH,
              ci_color=CI_COLOR, ci_linewidth=CI_LINEWIDTH, ci_tick=CI_TICK):
    '''
    Draw one prevalence ring on axes whose data units are mm.
    Inputs:
        xy: center (mm). k, n: ecDNA+ tumors and all tumors. ci: (low, high) as fractions.
        dark, light: colors of the ecDNA+ arc and the remainder.
        width: ring thickness (mm). ci_color, ci_linewidth (pt), ci_tick (end bar length as a fraction of width).
    Returns: outer radius (mm).
    '''
    r = ring_radius(n)
    p = k / n
    if p > 0:
        ax.add_patch(Wedge(xy, r, clock_angle(p), 90, width=width, facecolor=dark, edgecolor='none'))
    if p < 1:
        ax.add_patch(Wedge(xy, r, 90, clock_angle(p) + 360, width=width, facecolor=light, edgecolor='none'))
    # CI: arc through the middle of the ring, with radial end bars
    rm = r - width / 2
    lo, hi = ci
    ax.add_patch(Arc(xy, 2 * rm, 2 * rm, theta1=clock_angle(hi), theta2=clock_angle(lo),
                     color=ci_color, linewidth=ci_linewidth, capstyle='butt'))
    for q in (lo, hi):
        a = np.deg2rad(clock_angle(q))
        r0, r1 = rm - ci_tick * width / 2, rm + ci_tick * width / 2
        ax.plot([xy[0] + r0 * np.cos(a), xy[0] + r1 * np.cos(a)], [xy[1] + r0 * np.sin(a), xy[1] + r1 * np.sin(a)],
                color=ci_color, linewidth=ci_linewidth, solid_capstyle='butt')
    return r

def format_count(k, n):
    '''"(k/n, x%)" as in the previous Fig. 3: whole percent, or one decimal where that rounds below 1% (eg. 0.5%, but 1% for 1/101).'''
    k, n = int(k), int(n)
    p = k / n
    pct = f'{p:.1%}' if 0 < p < 0.0095 else f'{p:.0%}'
    return f'({k}/{n}, {pct})'

In [ ]:
def plot_ring(row, name, colors, label_side='right', size_mm=60):
    '''
    One ring with its label, on a figure whose data units are mm. The canvas is cropped to its contents on saving,
    which leaves the physical scale unchanged.
    Inputs:
        row: prevalence() row (k, n, ci_low, ci_high). name: display name (may contain line breaks).
        colors: {'dark', 'light'}. label_side: 'left', 'right', or None for no label.
    '''
    fig = plt.figure(figsize=(size_mm / 25.4, size_mm / 25.4))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, size_mm); ax.set_ylim(0, size_mm); ax.set_aspect('equal'); ax.axis('off')
    xy = (size_mm / 2, size_mm / 2)
    r = draw_ring(ax, xy, row.k, row.n, (row.ci_low, row.ci_high), colors['dark'], colors['light'])
    if label_side is not None:
        sign = 1 if label_side == 'right' else -1
        ax.text(xy[0] + sign * (r + LABEL_GAP), xy[1], f'{name}\n{format_count(row.k, row.n)}',
                ha='left' if label_side == 'right' else 'right', va='center',
                multialignment='left' if label_side == 'right' else 'right')
    return fig, ax

def save_ring(fig, basename, pad_mm=0.5):
    '''
    Save as SVG and PNG, cropped to the drawn ring and label plus pad_mm. Cropping only moves the canvas edges,
    so ring sizes stay in true mm. (bbox_inches='tight' would keep the whole canvas, since the axes fill it.)
    '''
    ax = fig.axes[0]
    renderer = fig.canvas.get_renderer()
    bbox = Bbox.union([a.get_window_extent(renderer) for a in ax.patches + ax.lines + ax.texts])
    bbox = bbox.transformed(fig.dpi_scale_trans.inverted()).padded(pad_mm / 25.4)
    for ext in ('svg', 'png'):
        fig.savefig(f'{basename}.{ext}', dpi=300, bbox_inches=bbox)
    plt.close(fig)

## One ring per tumor type

In [ ]:
for t, side in LABEL_SIDE.items():
    fig, ax = plot_ring(by_type.loc[t], TUMOR_TYPE_NAMES[t], get_colors(t), label_side=side)
    save_ring(fig, f'{OUTDIR}/{t}')
by_type.loc[list(LABEL_SIDE)]

Preview, in layout order.

In [ ]:
def plot_ring_sheet(df, columns=5, cell=(36, 16)):
    '''Contact sheet of rings with labels underneath, for checking. Not for the figure.'''
    rows = int(np.ceil(len(df) / columns))
    w, h = columns * cell[0], rows * cell[1]
    fig = plt.figure(figsize=(w / 25.4, h / 25.4))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, w); ax.set_ylim(0, h); ax.set_aspect('equal'); ax.axis('off')
    for i, (t, row) in enumerate(df.iterrows()):
        x = (i % columns + 0.5) * cell[0]
        y = h - (i // columns) * cell[1] - 5
        c = get_colors(t)
        draw_ring(ax, (x, y), row.k, row.n, (row.ci_low, row.ci_high), c['dark'], c['light'])
        ax.text(x, y - 5, f'{t} {format_count(row.k, row.n)}', ha='center', va='top')
    return fig, ax

fig, ax = plot_ring_sheet(by_type.loc[list(LABEL_SIDE)])

## Molecular subtypes
One ring per drawn subtype of each tumor type (`DRAWN_SUBTYPES`, or every subclass with at least `MIN_SUBTYPE_N` tumors), with the rest of the type pooled into an `OTHER_SUBTYPE` ring. Which subtypes go into the final figure is still to be decided.

In [ ]:
def assign_drawn_subtypes(tumors, drawn_subtypes=DRAWN_SUBTYPES, min_n=MIN_SUBTYPE_N, other=OTHER_SUBTYPE,
                          always=VALIDATION_SUBTYPES):
    '''
    Add a `drawn_subtype` column, the ring each tumor is counted in: its cancer_subclass if that subclass is drawn as
    its own ring, otherwise `other`. A type's own rings are drawn_subtypes[type] if listed, else every subclass with at
    least min_n tumors, plus any subtype key ('type.subclass') in `always`. Types with no own rings (no subclass
    annotation, or only `other`) get NaN.
    '''
    tumors = tumors.copy()
    key = tumors.cancer_type + '.' + tumors.cancer_subclass
    n = tumors.groupby(['cancer_type', 'cancer_subclass']).patient_id.transform('size') # NaN where no subclass
    listed = key.isin([f'{t}.{s}' for t, ss in drawn_subtypes.items() for s in ss])
    own = np.where(tumors.cancer_type.isin(list(drawn_subtypes)), listed, n >= min_n) | key.isin(always)
    tumors['drawn_subtype'] = tumors.cancer_subclass.where(own, other)
    has_own = pd.Series(own, index=tumors.index).groupby(tumors.cancer_type).transform('any')
    tumors.loc[~has_own, 'drawn_subtype'] = np.nan
    return tumors

def get_subtype_colors(cancer_type, drawn_subtype, colors=SUBTYPE_COLORS):
    '''Colors for a subtype ring: the SUBTYPE_COLORS override if present, otherwise its tumor type's colors.'''
    return get_colors(f'{cancer_type}.{drawn_subtype}', colors=colors, default=get_colors(cancer_type))

In [ ]:
# validation tumors count toward VALIDATION_TYPES and VALIDATION_SUBTYPES
subtype_key = tumors.cancer_type + '.' + tumors.cancer_subclass
subtype_tumors = assign_drawn_subtypes(tumors[(tumors.cohort == 'main') | tumors.cancer_type.isin(VALIDATION_TYPES) |
                                              subtype_key.isin(VALIDATION_SUBTYPES)])
by_subtype = prevalence(subtype_tumors.dropna(subset='drawn_subtype'), by=['cancer_type', 'drawn_subtype'])
# within each type: largest first, OTHER_SUBTYPE last
by_subtype = (by_subtype.reset_index()
              .assign(is_other=lambda df: df.drawn_subtype == OTHER_SUBTYPE)
              .sort_values(['cancer_type', 'is_other', 'n'], ascending=[True, True, False])
              .drop(columns='is_other').set_index(['cancer_type', 'drawn_subtype']))
by_subtype.to_csv(f'{OUTDIR}/prevalence_by_subtype.tsv', sep='\t')
by_subtype

In [ ]:
os.makedirs(f'{OUTDIR}/subtypes', exist_ok=True)
panel_subtypes = by_subtype.loc[[t for t in LABEL_SIDE if t in by_subtype.index.get_level_values('cancer_type')]]
for (t, s), row in panel_subtypes.iterrows():
    fig, ax = plot_ring(row, SUBTYPE_NAMES.get(f'{t}.{s}', s), get_subtype_colors(t, s), label_side=LABEL_SIDE[t])
    save_ring(fig, f'{OUTDIR}/subtypes/{t}.{s}')

Preview: each tumor type's ring, then its subtype rings.

In [ ]:
def plot_subtype_sheet(by_type, by_subtype, types, cell=(30, 20)):
    '''
    Contact sheet for checking: one row per tumor type, the type's ring first, then its subtype rings.
    Labels underneath give the subclass code and (k/n, %). Not for the figure.
    '''
    columns = 1 + max(len(by_subtype.loc[t]) for t in types)
    w, h = columns * cell[0], len(types) * cell[1]
    fig = plt.figure(figsize=(w / 25.4, h / 25.4))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, w); ax.set_ylim(0, h); ax.set_aspect('equal'); ax.axis('off')
    for i, t in enumerate(types):
        y = h - (i + 0.5) * cell[1] + 3
        rows = [(t, by_type.loc[t], get_colors(t))] + [(s, row, get_subtype_colors(t, s)) for s, row in by_subtype.loc[t].iterrows()]
        for j, (s, row, c) in enumerate(rows):
            x = (j + 0.5) * cell[0]
            draw_ring(ax, (x, y), row.k, row.n, (row.ci_low, row.ci_high), c['dark'], c['light'])
            ax.text(x, y - 5, f'{s}\n{format_count(row.k, row.n)}', ha='center', va='top',
                    fontweight='bold' if j == 0 else 'normal')
    ax.axvline(cell[0], color='#cccccc', linewidth=0.5)
    return fig, ax

fig, ax = plot_subtype_sheet(by_type, by_subtype, list(panel_subtypes.index.get_level_values('cancer_type').unique()))

## Figure panel composition
Each panel is drawn from `PANEL_LAYOUT`: rings, labels and leader lines at the layout's positions (mm from the panel's top left, y down), with the panel's silhouette from `assets/` inserted beneath them in the SVG.

In [ ]:
SVG_NS = 'http://www.w3.org/2000/svg'
for prefix, uri in [('', SVG_NS), ('xlink', 'http://www.w3.org/1999/xlink'),
                    ('inkscape', 'http://www.inkscape.org/namespaces/inkscape'),
                    ('sodipodi', 'http://sodipodi.sourceforge.net/DTD/sodipodi-0.dtd'),
                    ('dc', 'http://purl.org/dc/elements/1.1/'), ('cc', 'http://creativecommons.org/ns#'),
                    ('rdf', 'http://www.w3.org/1999/02/22-rdf-syntax-ns#')]:
    ET.register_namespace(prefix, uri)

def ring_data(key):
    '''Prevalence row, display name and colors for a layout key: a tumor type ('HGG') or a subtype ('EPN.MYCN').'''
    if '.' in key:
        t, s = key.split('.', 1)
        return by_subtype.loc[(t, s)], SUBTYPE_NAMES.get(key, s), get_subtype_colors(t, s)
    return by_type.loc[key], TUMOR_TYPE_NAMES[key], get_colors(key)

def compose_panel(panel, letter=None, label_gap=LABEL_GAP):
    '''
    Draw a panel's rings, labels and leader lines on a figure whose data units are mm. Layout coordinates are from the
    panel's top left with y down, as in Inkscape; they are flipped here so draw_ring still runs clockwise from 12 o'clock.
    The silhouette is added by save_panel.
    Inputs:
        panel: one entry of the layout's panels. letter: panel letter, drawn at the panel's letter_xy (None to omit).
    '''
    w, h = panel['size']
    flip = lambda p: (p[0], h - p[1])
    fig = plt.figure(figsize=(w / 25.4, h / 25.4))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, w); ax.set_ylim(0, h); ax.set_aspect('equal'); ax.axis('off')
    if letter is not None:
        ax.text(*flip(panel['letter_xy']), letter, fontsize=LETTER_FONTSIZE, fontweight='bold', va='baseline')
    for key, ring in panel['rings'].items():
        row, name, colors = ring_data(key)
        if 'line' in ring:
            x, y = zip(*map(flip, ring['line']))
            ax.plot(x, y, color='black', linewidth=LEADER_LINEWIDTH, solid_capstyle='butt', solid_joinstyle='miter')
        xy = flip(ring['center'])
        r = draw_ring(ax, xy, row.k, row.n, (row.ci_low, row.ci_high), colors['dark'], colors['light'])
        sign = 1 if ring['label'] == 'right' else -1
        ax.text(xy[0] + sign * (r + label_gap), xy[1], f'{name}\n{format_count(row.k, row.n)}',
                ha='left' if ring['label'] == 'right' else 'right', va='center',
                multialignment='left' if ring['label'] == 'right' else 'right')
    return fig, ax

def insert_silhouette(svg_path, panel, root='../..'):
    '''
    Insert the panel's silhouette SVG beneath the matplotlib drawing, with its top left corner at silhouette_xy (mm).
    The silhouette keeps its own size: its width attribute (mm) over its viewBox width sets the scale.
    '''
    pt = 72 / 25.4 # matplotlib writes SVG in points
    tree = ET.parse(svg_path)
    svg = tree.getroot()
    asset = ET.parse(os.path.join(root, panel['silhouette'])).getroot()
    vb = [float(v) for v in asset.get('viewBox').split()]
    mm_per_unit = float(asset.get('width').removesuffix('mm')) / vb[2]
    x, y = panel['silhouette_xy']
    g = ET.Element(f'{{{SVG_NS}}}g', {
        'id': 'silhouette',
        'transform': f'translate({x * pt:.4f} {y * pt:.4f}) scale({mm_per_unit * pt:.6f}) translate({-vb[0]} {-vb[1]})'})
    for child in asset:
        if child.tag in (f'{{{SVG_NS}}}g', f'{{{SVG_NS}}}path', f'{{{SVG_NS}}}defs'):
            g.append(child)
    # directly before the matplotlib drawing, so the silhouette sits beneath its rings and lines
    svg.insert(next(i for i, c in enumerate(svg) if c.get('id') == 'figure_1'), g)
    tree.write(svg_path, xml_declaration=True, encoding='utf-8')

def save_panel(fig, panel, basename, inkscape=INKSCAPE, dpi=300):
    '''
    Save the panel as SVG (uncropped, so layout coordinates hold, and with a transparent background so the silhouette
    shows through), insert its silhouette, and render a PNG with Inkscape.
    '''
    fig.savefig(f'{basename}.svg', transparent=True)
    plt.close(fig)
    insert_silhouette(f'{basename}.svg', panel)
    if inkscape and shutil.which(inkscape):
        subprocess.run([inkscape, f'{basename}.svg', '--export-type=png', f'--export-dpi={dpi}',
                        '--export-background=white', f'--export-filename={basename}.png'], check=True, capture_output=True)

In [ ]:
for name, panel in panel_layout['panels'].items():
    fig, ax = compose_panel(panel, letter=panel.get('letter'), label_gap=panel_layout.get('label_gap', LABEL_GAP))
    save_panel(fig, panel, f'{OUTDIR}/panel_{name}')